# 📊 FakeInversion - Notebook 4: Evaluation & Visualization

This notebook:
1. Evaluates detector on test set (same distribution)
2. Evaluates generalization across ALL generator models
3. Generates ROC curves, accuracy heatmaps
4. Compares results with the original paper

In [ ]:
# Setup
import os, sys
PROJECT_DIR = '/content/fake_inversion'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from google.colab import drive
drive.mount('/content/drive')

import config
config.print_config()

In [ ]:
# Full evaluation
from evaluation.evaluate import run_full_evaluation, Evaluator

run_full_evaluation()

In [ ]:
# Generate all plots
from evaluation.visualize import generate_all_plots
import matplotlib.pyplot as plt

generate_all_plots()

# Display plots
plots_dir = os.path.join(config.RESULTS_DIR, 'plots')
for plot_file in sorted(os.listdir(plots_dir)):
    if plot_file.endswith('.png'):
        img = plt.imread(os.path.join(plots_dir, plot_file))
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img)
        ax.set_title(plot_file, fontsize=14)
        ax.axis('off')
        plt.show()

In [ ]:
# Load and display detailed results
import json
import pandas as pd

# Per-model results
results_path = os.path.join(config.RESULTS_DIR, 'per_model_results.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    
    rows = []
    for model, metrics in results.items():
        if model.startswith('_'):
            continue
        is_train = '✓' if model == config.TRAIN_SOURCE_MODEL else ''
        rows.append({
            'Model': model,
            'Train': is_train,
            'Samples': metrics.get('num_samples', 0),
            'Accuracy': f"{metrics['accuracy']:.4f}",
            'AUC': f"{metrics['auc']:.4f}",
            'AP': f"{metrics['ap']:.4f}",
        })
    
    df = pd.DataFrame(rows)
    print('\n📊 Per-Model Detection Results')
    print('=' * 70)
    print(df.to_string(index=False))
    print('=' * 70)
    
    avg = results.get('_average', {})
    print(f"\nAverage Accuracy: {avg.get('accuracy', 0):.4f}")
    print(f"Average AUC:      {avg.get('auc', 0):.4f}")
    print(f"Average AP:       {avg.get('ap', 0):.4f}")

In [ ]:
# Demo: Test on a single new image
import torch
from PIL import Image
from inversion.ddim_inverter import DDIMInverter
from classifier.model import FakeInversionClassifier
from classifier.dataset import EvalTransform
from utils import load_checkpoint
import torchvision.transforms as T

def detect_single_image(image_path: str):
    """Run FakeInversion detection on a single image."""
    device = config.get_device()
    
    # Extract features
    inverter = DDIMInverter()
    image = Image.open(image_path).convert('RGB')
    features = inverter.extract_features(image)  # (9, H, W)
    
    # Load classifier
    model = FakeInversionClassifier().to(device)
    ckpt = load_checkpoint(
        os.path.join(config.CHECKPOINTS_DIR, 'best_model.pt'),
        map_location=device
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    # Preprocess
    transform = EvalTransform()
    features = transform(features)
    features = T.functional.resize(features, (224, 224), antialias=True)
    features = features.unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        probs = model.predict_proba(features)
    
    real_prob = probs[0, 0].item()
    fake_prob = probs[0, 1].item()
    
    print(f'\n🔍 Detection Result for: {os.path.basename(image_path)}')
    print(f'   Real probability: {real_prob:.4f}')
    print(f'   Fake probability: {fake_prob:.4f}')
    print(f'   Verdict: {"🟢 REAL" if real_prob > fake_prob else "🔴 FAKE"}')
    
    # Cleanup
    inverter.unload_models()
    return {'real': real_prob, 'fake': fake_prob}

# Example usage:
# result = detect_single_image('/path/to/your/image.png')

## ✅ Evaluation complete!

### Summary
- **Training**: Detector trained ONLY on SD-1.5 fakes
- **Generalization**: Tested on ALL 15 generator models
- **Key insight**: Inversion features allow generalization to unseen models

### Files saved to Google Drive:
- `results/per_model_results.json` - Detailed per-model metrics
- `results/roc_data.json` - ROC curve data
- `results/plots/` - All visualization plots
- `checkpoints/best_model.pt` - Trained model